In [6]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import umap
import biom
from qiime2 import Metadata
from gemelli.preprocessing import matrix_rclr
import os
import time
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import qiime2 as q2
from qiime2 import Artifact, Metadata, Visualization
from qiime2.plugins.sample_classifier.pipelines import classify_samples, heatmap

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


All libraries imported successfully!
PyTorch version: 1.12.1
CUDA available: False


In [8]:
# Dictionary to store all dataframes
dataframes = {}

# Loop through files s1 to s15
for i in range(1, 16):
    filename = f'data/TCGA_Microbial_Content/data_file_s{i}.xlsx'
    df_name = f's{i}'
    
    try:
        # Read the Excel file
        dataframes[df_name] = pd.read_excel(filename)
        print(f"Successfully loaded {filename} as '{df_name}'")
    except FileNotFoundError:
        print(f"Warning: {filename} not found")
    except Exception as e:
        print(f"Error loading {filename}: {str(e)}")

# dataframes['s1'], dataframes['s2']

s1 = dataframes.get('s1')   # Solid tissue normal
s2 = dataframes.get('s2')   # Blood-derived normal
s3 = dataframes.get('s3')   # Raw counts - Microbial2023 species
s4 = dataframes.get('s4')   # Normalized counts - Microbial2023 species
s5 = dataframes.get('s5')   # Raw counts - Fungi species
s6 = dataframes.get('s6')   # Normalized counts - Fungi species
s7 = dataframes.get('s7')   # Viral counts (18 viruses)
s8 = dataframes.get('s8')   # Raw counts - genus level (this study)
s9 = dataframes.get('s9')   # Raw counts - genus level (Poore et al.)
s10 = dataframes.get('s10') # Raw counts - Fungi (matched with Narunsky-Haziza)
s11 = dataframes.get('s11') # Raw counts - Fungi (Narunsky-Haziza et al.)
s12 = dataframes.get('s12') # Metadata
s13 = dataframes.get('s13') # Microbial species list
s14 = dataframes.get('s14') # Fungi species list
s15 = dataframes.get('s15') # Name conversion list


if s1 is not None:
    print("\ns1 dataframe info:")
    print(s1.head())
    print(f"\nShape: {s1.shape}")

Successfully loaded data/TCGA_Microbial_Content/data_file_s1.xlsx as 's1'
Successfully loaded data/TCGA_Microbial_Content/data_file_s2.xlsx as 's2'
Successfully loaded data/TCGA_Microbial_Content/data_file_s3.xlsx as 's3'
Successfully loaded data/TCGA_Microbial_Content/data_file_s4.xlsx as 's4'
Successfully loaded data/TCGA_Microbial_Content/data_file_s5.xlsx as 's5'
Successfully loaded data/TCGA_Microbial_Content/data_file_s6.xlsx as 's6'
Successfully loaded data/TCGA_Microbial_Content/data_file_s7.xlsx as 's7'
Successfully loaded data/TCGA_Microbial_Content/data_file_s8.xlsx as 's8'
Successfully loaded data/TCGA_Microbial_Content/data_file_s9.xlsx as 's9'
Successfully loaded data/TCGA_Microbial_Content/data_file_s10.xlsx as 's10'
Successfully loaded data/TCGA_Microbial_Content/data_file_s11.xlsx as 's11'
Successfully loaded data/TCGA_Microbial_Content/data_file_s12.xlsx as 's12'
Successfully loaded data/TCGA_Microbial_Content/data_file_s13.xlsx as 's13'
Successfully loaded data/TCGA_

In [14]:
import pandas as pd
import numpy as np

# File descriptions for reference
file_descriptions = {
    's1': 'Solid tissue normal samples (569 samples)',
    's2': 'Blood-derived normal samples (2,277 samples)',
    's3': 'Raw counts - species level - Microbial2023 (5,734 samples, 11,332 species)',
    's4': 'Normalized counts (CPM) - species level - Microbial2023',
    's5': 'Raw counts - species level - Fungi_RefSeq (5,734 samples, 557 species)',
    's6': 'Normalized counts (CPM) - species level - Fungi_RefSeq',
    's7': 'Raw counts - 18 selected viruses (5,734 samples)',
    's8': 'Raw counts - genus level - Microbial2023 (4,550 samples, 3,563 genera)',
    's9': 'Raw counts - genus level - Poore et al. comparison (4,550 samples)',
    's10': 'Raw counts - species level - Fungi_RefSeq matched (4,271 samples, 224 species)',
    's11': 'Raw counts - species level - Narunsky-Haziza et al. (4,271 samples)',
    's12': 'TCGA metadata (5,734 samples)',
    's13': 'Microbial2023 species list with RefSeq accessions',
    's14': 'Fungi_RefSeq species list with RefSeq accessions',
    's15': 'RefSeq v200 to v220 fungal name conversion (14 species)'
}


In [15]:

def preprocess_sample_overview(df, sample_type):
    """Preprocess sample overview data (s1, s2)"""
    if df is None:
        return None
    
    df_clean = df.copy()
    print(f"\n{sample_type} Overview:")
    print(f"  - Shape: {df_clean.shape}")
    print(f"  - Columns: {list(df_clean.columns)}")
    
    # Check for missing values
    missing = df_clean.isnull().sum()
    if missing.sum() > 0:
        print(f"  - Missing values detected:")
        print(missing[missing > 0])
    
    return df_clean

def preprocess_count_matrix(df, data_type, is_normalized=False):
    """Preprocess count matrices (s3-s6, s8-s11)"""
    if df is None:
        return None
    
    df_clean = df.copy()
    count_type = "CPM" if is_normalized else "Raw counts"
    
    print(f"\n{data_type} ({count_type}):")
    print(f"  - Shape: {df_clean.shape}")
    
    # Identify sample columns (typically everything except first column which is species/genus)
    if df_clean.shape[1] > 1:
        id_col = df_clean.columns[0]
        sample_cols = df_clean.columns[1:]
        
        print(f"  - ID column: {id_col}")
        print(f"  - Number of samples: {len(sample_cols)}")
        print(f"  - Number of taxa: {len(df_clean)}")
        
        # Check for negative values (shouldn't exist in count data)
        numeric_data = df_clean[sample_cols].select_dtypes(include=[np.number])
        if (numeric_data < 0).any().any():
            print(f"  ⚠ Warning: Negative values detected!")
        
        # Summary statistics
        total_counts = numeric_data.sum().sum()
        print(f"  - Total counts across all samples: {total_counts:,.0f}")
        print(f"  - Mean counts per sample: {numeric_data.sum().mean():,.2f}")
        print(f"  - Median counts per sample: {numeric_data.sum().median():,.2f}")
    
    return df_clean

def preprocess_metadata(df):
    """Preprocess metadata (s12)"""
    if df is None:
        return None
    
    df_clean = df.copy()
    print(f"\nMetadata:")
    print(f"  - Shape: {df_clean.shape}")
    print(f"  - Columns: {list(df_clean.columns)}")
    
    # Check for duplicates
    if df_clean.shape[1] > 0:
        first_col = df_clean.columns[0]
        duplicates = df_clean[first_col].duplicated().sum()
        if duplicates > 0:
            print(f"  ⚠ Warning: {duplicates} duplicate IDs detected!")
    
    return df_clean

def preprocess_reference_list(df, ref_type):
    """Preprocess reference lists (s13-s15)"""
    if df is None:
        return None
    
    df_clean = df.copy()
    print(f"\n{ref_type} Reference List:")
    print(f"  - Shape: {df_clean.shape}")
    print(f"  - Columns: {list(df_clean.columns)}")
    
    return df_clean


In [16]:

# Sample overviews
s1_clean = preprocess_sample_overview(s1, "Solid Tissue Normal")
s2_clean = preprocess_sample_overview(s2, "Blood-Derived Normal")

# Count matrices - Microbial
s3_clean = preprocess_count_matrix(s3, "Microbial2023 Species", is_normalized=False)
s4_clean = preprocess_count_matrix(s4, "Microbial2023 Species", is_normalized=True)

# Count matrices - Fungi
s5_clean = preprocess_count_matrix(s5, "Fungi Species", is_normalized=False)
s6_clean = preprocess_count_matrix(s6, "Fungi Species", is_normalized=True)

# Viral counts
s7_clean = preprocess_count_matrix(s7, "Viral Counts (18 viruses)", is_normalized=False)

# Genus level comparisons
s8_clean = preprocess_count_matrix(s8, "Genus Level - This Study", is_normalized=False)
s9_clean = preprocess_count_matrix(s9, "Genus Level - Poore et al.", is_normalized=False)

# Fungi comparisons
s10_clean = preprocess_count_matrix(s10, "Fungi - This Study (matched)", is_normalized=False)
s11_clean = preprocess_count_matrix(s11, "Fungi - Narunsky-Haziza et al.", is_normalized=False)

# Metadata and reference lists
s12_clean = preprocess_metadata(s12)
s13_clean = preprocess_reference_list(s13, "Microbial2023 Species")
s14_clean = preprocess_reference_list(s14, "Fungi Species")
s15_clean = preprocess_reference_list(s15, "Fungal Name Conversion")



Solid Tissue Normal Overview:
  - Shape: (22, 10)
  - Columns: ['Cancer type', 'Total # samples', 'Average read count (millions)', 'Unmapped reads after mapping to GRCh38 (avg, thousands)', 'Unnamed: 4', 'Unmapped reads after mapping to GRCh38+CHM13 (avg, thousands)', 'Unnamed: 6', 'Kraken-identified human reads (avg, thousands)', 'Unnamed: 8', 'Kraken-identified vector reads (avg, thousands)']
  - Missing values detected:
Total # samples                                                   2
Average read count (millions)                                     2
Unmapped reads after mapping to GRCh38 (avg, thousands)           2
Unnamed: 4                                                       22
Unmapped reads after mapping to GRCh38+CHM13 (avg, thousands)     2
Unnamed: 6                                                       22
Kraken-identified human reads (avg, thousands)                    2
Unnamed: 8                                                       22
Kraken-identified vector rea

In [13]:

# Store cleaned dataframes
cleaned_data = {
    's1': s1_clean, 's2': s2_clean, 's3': s3_clean, 's4': s4_clean,
    's5': s5_clean, 's6': s6_clean, 's7': s7_clean, 's8': s8_clean,
    's9': s9_clean, 's10': s10_clean, 's11': s11_clean, 's12': s12_clean,
    's13': s13_clean, 's14': s14_clean, 's15': s15_clean
}
